# 02: ML-Powered Prompt Generation

This notebook explores the idea of using a text-generation model (like GPT-2) to create rich, descriptive prompts for our audio generation model (MusicGen).

**Goal:** To automate the creative writing process and generate more complex and evocative prompts than our manual agents.

## 1. Imports and Setup

We'll start by importing the `pipeline` from the `transformers` library, which we've already installed.

In [5]:
from transformers import pipeline
import torch

## 2. Initialize the Text Generation Pipeline

We'll load a pre-trained text-generation model. `gpt2-medium` is a more powerful model than `distilgpt2` and should give us more coherent results. The first time you run this cell, it will download the new, larger model.

In [6]:
prompt_generator = pipeline('text-generation', model='gpt2-medium')

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

## 3. Create a "Meta-Prompt"

Now, we create a prompt *for our prompt generator*. This "meta-prompt" will guide the model on what kind of description we want. We'll use 'role-playing' to encourage more creative results.

In [7]:
theme = "Water"
scientific_elements = "a 10Hz alpha wave binaural beat and a subtle layer of pink noise"

# By instructing the model to adopt a persona, we can often get more creative results.
meta_prompt = f"You are a creative sound designer tasked with creating a soundscape for a popular YouTube channel that helps people focus and relax. Write an evocative, one-paragraph description of a soundscape based on the theme of '{theme}'. The description must include the phrases '{scientific_elements}'. The tone should be inspiring and descriptive, focusing on the feeling and texture of the sound."

print("--- META-PROMPT ---")
print(meta_prompt)

--- META-PROMPT ---
You are a creative sound designer tasked with creating a soundscape for a popular YouTube channel that helps people focus and relax. Write an evocative, one-paragraph description of a soundscape based on the theme of 'Water'. The description must include the phrases 'a 10Hz alpha wave binaural beat and a subtle layer of pink noise'. The tone should be inspiring and descriptive, focusing on the feeling and texture of the sound.


## 4. Generate the Rich Prompt

Let's run the meta-prompt through our generator. We'll add a `temperature` setting to encourage more creative, less predictable output.

In [8]:
generated_output = prompt_generator(
    meta_prompt,
    max_new_tokens=75,  # Increased token limit for more descriptive text
    num_return_sequences=1,
    temperature=0.8,    # Higher temperature for more creativity
    do_sample=True      # Required for temperature to have an effect
)

# --- Output Cleaning ---
full_text = generated_output[0]['generated_text']

# 1. Isolate the newly generated part by removing the meta-prompt
newly_generated_text = full_text.replace(meta_prompt, "").strip()

# 2. Find the first sensible sentence from the new part.
import re
first_sentence = re.split(r'(?<=[.!?])\s+', newly_generated_text)[0]

# 3. Combine the original instructions with the new creative sentence.
final_rich_prompt = meta_prompt + " " + first_sentence

print("--- FINAL, CLEANED RICH PROMPT ---")
print(final_rich_prompt)

Passing `generation_config` together with generation-related arguments=({'do_sample', 'num_return_sequences', 'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=75) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- FINAL, CLEANED RICH PROMPT ---
You are a creative sound designer tasked with creating a soundscape for a popular YouTube channel that helps people focus and relax. Write an evocative, one-paragraph description of a soundscape based on the theme of 'Water'. The description must include the phrases 'a 10Hz alpha wave binaural beat and a subtle layer of pink noise'. The tone should be inspiring and descriptive, focusing on the feeling and texture of the sound. 'Water' has earned over 1.3 million views.


## Next Steps

The text printed above is the new, rich prompt that we can now feed into our MusicGen audio generation script!

We can experiment by:
*   Changing the `meta_prompt` with different themes and elements.
*   Trying different text-generation models.
*   Integrating this into a script that first generates a prompt, then generates the audio from that prompt.